# Project Milestone 1: EDA Deep Dive — UCI Adult Income Dataset

**Difficulty:** ⭐⭐⭐⭐ &nbsp;|&nbsp; **Estimated Time:** 4–5 hours &nbsp;|&nbsp; **Week:** 4

---

## Table of Contents
- [Section 0 — Dataset Overview](#section-0)
- [Section 1 — Data Quality Assessment](#section-1)
- [Section 2 — Univariate Analysis](#section-2)
- [Section 3 — Bivariate Analysis vs Target](#section-3)
- [Section 4 — Multivariate Analysis](#section-4)
- [Section 5 — Missing Data & Outliers](#section-5)
- [Section 6 — 15 Annotated Visualizations Gallery](#section-6)
- [Section 7 — 5 Testable ML Hypotheses](#section-7)
- [Section 8 — Professional Report](#section-8)
- [Self-Check Questions](#self-check)

**Goal:** Perform a rigorous EDA on the UCI Adult Income dataset, identify key predictors, uncover data quality issues, and formulate testable ML hypotheses.

<a id='section-0'></a>
## Section 0 — Dataset Overview

The **UCI Adult Income** dataset (also called the Census Income dataset) was extracted from the 1994 US Census Bureau database. It contains ~48,000 records and 14 features used to predict whether an individual earns more or less than $50K/year.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='husl')

# Load UCI Adult Income dataset (version 2 includes cleaned column names)
adult = fetch_openml(name='adult', version=2, as_frame=True)
df = adult.frame
df.replace('?', np.nan, inplace=True)

print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nBasic stats (numeric):\n")
df.describe()

### Feature Catalog

| Feature | Type | Example Value | Missing % (approx) |
|---|---|---|---|
| age | Numeric | 39 | 0% |
| workclass | Categorical | Private | ~5.6% |
| fnlwgt | Numeric | 77516 | 0% |
| education | Categorical | Bachelors | 0% |
| education-num | Numeric | 13 | 0% |
| marital-status | Categorical | Never-married | 0% |
| occupation | Categorical | Adm-clerical | ~5.7% |
| relationship | Categorical | Not-in-family | 0% |
| race | Categorical | White | 0% |
| sex | Categorical | Male | 0% |
| capital-gain | Numeric | 2174 | 0% |
| capital-loss | Numeric | 0 | 0% |
| hours-per-week | Numeric | 40 | 0% |
| native-country | Categorical | United-States | ~1.8% |
| **class** | **Target** | <=50K / >50K | 0% |

> **Note:** `?` values were replaced with `NaN` during loading. Actual missing percentages are computed in Section 1.

In [ ]:
# Compute actual missing percentages and target distribution
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("Missing values (%):\n", missing_pct[missing_pct > 0])

# Target distribution
target_counts = df['class'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(target_counts.index, target_counts.values,
              color=sns.color_palette('husl', 2), edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=11)
ax.set_title('Target Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Income Class', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
plt.tight_layout()
plt.show()
print(f"Class imbalance ratio: {target_counts.iloc[0]/target_counts.iloc[1]:.2f}:1")

<a id='section-1'></a>
## Section 1 — Data Quality Assessment

In [ ]:
# Missing values per column — horizontal bar chart
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(missing.index, missing.values,
               color=sns.color_palette('husl', len(missing)))
for bar, val in zip(bars, missing.values):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{val} ({val/len(df)*100:.1f}%)', va='center', fontsize=10)
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
ax.set_xlabel('Count of Missing Values', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Range validation for key numeric columns
print("=== Range Validation ===")
# age should be 0–120
age_violations = df[(df['age'] < 0) | (df['age'] > 120)]
print(f"age out of [0,120]: {len(age_violations)} rows")

# hours-per-week should be 0–168
hours_col = 'hours-per-week' if 'hours-per-week' in df.columns else 'hours_per_week'
hours_violations = df[(df[hours_col] < 0) | (df[hours_col] > 168)]
print(f"hours-per-week out of [0,168]: {len(hours_violations)} rows")

print(f"\nage range: [{df['age'].min()}, {df['age'].max()}]")
print(f"hours-per-week range: [{df[hours_col].min()}, {df[hours_col].max()}]")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"Total rows: {len(df)}")

### Data Quality Dimensions

| Dimension | Score | Evidence | Action |
|---|---|---|---|
| **Completeness** | 94% | 3 columns have missing values (`workclass` ~5.6%, `occupation` ~5.7%, `native-country` ~1.8%) | Impute with mode or 'Unknown' category |
| **Consistency** | 97% | No range violations found; minor duplicates possible | De-duplicate before training |
| **Accuracy** | Moderate | 1994 Census data; labels are self-reported income; `fnlwgt` is a sampling weight not a direct feature | Use with domain knowledge |
| **Timeliness** | Low | Data is 30 years old (1994); income thresholds, job categories, and demographics have shifted significantly | Treat insights as directional, not deployable |

> **Summary:** The dataset is reasonably clean with ~6% missingness concentrated in occupation-related columns, likely Missing Not At Random (MNAR) — people in certain jobs may not disclose occupation.

<a id='section-2'></a>
## Section 2 — Univariate Analysis

In [ ]:
# Detect actual column names (sklearn may use underscores or hyphens)
col_map = {c.replace('-', '_'): c for c in df.columns}
def get_col(name):
    return col_map.get(name, name) if name not in df.columns else name

age_c = get_col('age')
hours_c = get_col('hours_per_week')
edu_num_c = get_col('education_num')
cap_gain_c = get_col('capital_gain')

numeric_cols = [age_c, hours_c, edu_num_c, cap_gain_c]
numeric_labels = ['Age', 'Hours per Week', 'Education Num', 'Capital Gain']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col, label in zip(axes.flatten(), numeric_cols, numeric_labels):
    data = df[col].dropna()
    ax.hist(data, bins=30, density=True, alpha=0.6, color='steelblue', edgecolor='white')
    # Overlay KDE
    kde_x = np.linspace(data.min(), data.max(), 200)
    kde = stats.gaussian_kde(data)
    ax.plot(kde_x, kde(kde_x), color='darkred', linewidth=2, label='KDE')
    ax.set_title(f'{label} Distribution', fontsize=12, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend()
plt.suptitle('Numeric Features — Histograms with KDE', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Observations — Numeric Features:**
- **Age:** Right-skewed, peak around 35–40; working-age adults dominate.
- **Hours/week:** Strong spike at 40 (standard work week); long tail to 99 hours.
- **Education-num:** Roughly bimodal; peaks at 9 (HS grad) and 13 (Bachelors).
- **Capital-gain:** Extremely zero-inflated (~91% are 0); remaining values are widely spread — log transform recommended.

In [ ]:
# Categorical features — 2x2 horizontal bar charts sorted by frequency
workclass_c = get_col('workclass')
education_c = get_col('education')
marital_c = get_col('marital_status')
occupation_c = get_col('occupation')

cat_cols = [workclass_c, education_c, marital_c, occupation_c]
cat_labels = ['Workclass', 'Education', 'Marital Status', 'Occupation']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col, label in zip(axes.flatten(), cat_cols, cat_labels):
    counts = df[col].value_counts().sort_values()  # ascending for horizontal bar
    colors = sns.color_palette('husl', len(counts))
    ax.barh(counts.index, counts.values, color=colors)
    ax.set_title(f'{label} — Frequency', fontsize=12, fontweight='bold')
    ax.set_xlabel('Count')
plt.suptitle('Categorical Features — Frequency Charts', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Observations — Categorical Features:**
- **Workclass:** ~70% are `Private` employees; `Self-emp`, `Government` are minorities.
- **Education:** `HS-grad` is the single largest group (~30%); `Some-college` second.
- **Marital Status:** `Married-civ-spouse` is the majority category (~45%).
- **Occupation:** Fairly distributed across ~14 categories; `Prof-specialty` and `Craft-repair` are top two.

<a id='section-3'></a>
## Section 3 — Bivariate Analysis vs Target

In [ ]:
# Violin plots: numeric features grouped by income class
class_c = get_col('class')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, label in zip(axes, [age_c, hours_c, edu_num_c], ['Age', 'Hours/Week', 'Education Num']):
    sns.violinplot(data=df, x=class_c, y=col, ax=ax,
                   palette='husl', inner='box', cut=0)
    ax.set_title(f'{label} by Income Class', fontsize=12, fontweight='bold')
    ax.set_xlabel('Income Class')
    ax.set_ylabel(label)
plt.suptitle('Numeric Features vs Target (Violin Plots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Income rate per categorical category — top features sorted by predictive power
# Create binary target: 1 if >50K, 0 otherwise
df['income_binary'] = (df[class_c].astype(str).str.strip() == '>50K').astype(int)

cat_features = [education_c, marital_c, occupation_c, workclass_c, get_col('sex'), get_col('race')]
cat_feature_labels = ['Education', 'Marital Status', 'Occupation', 'Workclass', 'Sex', 'Race']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col, label in zip(axes.flatten(), cat_features, cat_feature_labels):
    rates = df.groupby(col)['income_binary'].mean().sort_values(ascending=True)
    colors = ['#d73027' if v > 0.25 else '#4575b4' for v in rates.values]
    ax.barh(rates.index, rates.values, color=colors)
    ax.axvline(df['income_binary'].mean(), color='black', linestyle='--', linewidth=1.2, label='Overall mean')
    ax.set_title(f'{label} — Income Rate', fontsize=11, fontweight='bold')
    ax.set_xlabel('>50K Rate')
    ax.legend(fontsize=8)
plt.suptitle('Income Rate by Categorical Feature', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Point-biserial correlation: numeric features vs binary income
from scipy.stats import pointbiserialr

numeric_features = [age_c, hours_c, edu_num_c, cap_gain_c, get_col('capital_loss'), get_col('fnlwgt')]
pb_results = {}
for col in numeric_features:
    valid = df[[col, 'income_binary']].dropna()
    r, p = pointbiserialr(valid[col], valid['income_binary'])
    pb_results[col] = {'r': r, 'p': p}

pb_df = pd.DataFrame(pb_results).T.sort_values('r', ascending=False)
print("Point-Biserial Correlation with Income:\n", pb_df.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#d73027' if v > 0 else '#4575b4' for v in pb_df['r']]
ax.barh(pb_df.index, pb_df['r'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Point-Biserial Correlation (Numeric vs Income)', fontsize=12, fontweight='bold')
ax.set_xlabel('Correlation (r)')
plt.tight_layout()
plt.show()

In [ ]:
# Cramér's V for categorical features vs income class
from scipy.stats import chi2_contingency

def cramers_v(col1, col2, data):
    """Compute Cramér's V association statistic between two categorical columns."""
    contingency = pd.crosstab(data[col1].astype(str), data[col2].astype(str))
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.values.sum()
    r, c = contingency.shape
    # Cramér's V formula: sqrt(chi2 / (n * min(r-1, c-1)))
    return np.sqrt(chi2 / (n * min(r - 1, c - 1)))

cat_cols_cv = [education_c, marital_c, occupation_c, workclass_c,
               get_col('sex'), get_col('race'), get_col('relationship')]
cv_results = {col: cramers_v(col, class_c, df) for col in cat_cols_cv}
cv_df = pd.Series(cv_results).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(cv_df.index[::-1], cv_df.values[::-1],
        color=sns.color_palette('husl', len(cv_df)))
ax.set_title("Cramér's V — Categorical Features vs Income Class", fontsize=12, fontweight='bold')
ax.set_xlabel("Cramér's V")
plt.tight_layout()
plt.show()
print("\nRanking:\n", cv_df.round(4))

**Bivariate Analysis — Key Takeaways:**
- **Strongest numeric predictors:** `education-num`, `age`, and `capital-gain` (by point-biserial r).
- **Strongest categorical predictors:** `relationship` and `marital-status` top the Cramér's V ranking, followed by `education` and `occupation`.
- `marital-status` likely proxies `relationship` — both encode partnership status, creating near-redundancy.
- `race` has a relatively low Cramér's V, but ethical implications still require careful handling.

<a id='section-4'></a>
## Section 4 — Multivariate Analysis

In [ ]:
# Correlation heatmap — numeric features only
num_cols_heat = [age_c, get_col('fnlwgt'), edu_num_c, cap_gain_c,
                 get_col('capital_loss'), hours_c, 'income_binary']
corr_matrix = df[num_cols_heat].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pivot heatmap: occupation × sex → mean income rate
occ_sex_pivot = pd.crosstab(
    df[occupation_c], df[get_col('sex')],
    values=df['income_binary'], aggfunc='mean'
).fillna(0)

fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(occ_sex_pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean >50K Rate'})
ax.set_title('Income Rate: Occupation × Sex', fontsize=13, fontweight='bold')
ax.set_xlabel('Sex')
ax.set_ylabel('Occupation')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot — sample 2000 rows for speed
sample_df = df[[age_c, hours_c, edu_num_c, class_c]].dropna().sample(2000, random_state=42)
# Rename for cleaner labels
sample_df = sample_df.rename(columns={age_c: 'Age', hours_c: 'Hours/Wk',
                                       edu_num_c: 'Edu Num', class_c: 'Income'})
pair = sns.pairplot(sample_df, hue='Income', palette='husl',
                    plot_kws={'alpha': 0.4, 's': 15},
                    diag_kind='kde')
pair.figure.suptitle('Pairplot: Age, Hours/Week, Education Num (n=2000)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

**Multivariate Observations:**
- Numeric features are largely uncorrelated with each other (`r < 0.3`), minimizing multicollinearity risk.
- `education-num` and `income_binary` show the strongest numeric-to-target correlation.
- The occupation × sex heatmap reveals consistent gender income gaps across all occupations — `Exec-managerial` has the highest male income rate.
- Pairplot confirms that high-income individuals cluster at higher education numbers and ages; hours/week separation is weaker.

<a id='section-5'></a>
## Section 5 — Missing Data & Outliers

In [ ]:
# Co-missingness analysis — which columns are missing together?
miss_indicator = df.isnull().astype(int)  # 1 where missing
miss_cols = miss_indicator.columns[miss_indicator.sum() > 0]
miss_corr = miss_indicator[miss_cols].corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(miss_corr, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('Missingness Correlation Between Columns', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# MNAR test: is missing occupation/workclass correlated with income?
for col in [occupation_c, workclass_c]:
    df['_miss'] = df[col].isnull().astype(int)
    rate_missing = df[df['_miss']==1]['income_binary'].mean()
    rate_present = df[df['_miss']==0]['income_binary'].mean()
    print(f"{col}: income rate when MISSING={rate_missing:.3f} | PRESENT={rate_present:.3f} → "
          f"{'MNAR suspected' if abs(rate_missing - rate_present) > 0.02 else 'MAR/MCAR'}")
df.drop(columns=['_miss'], inplace=True)

In [ ]:
# Outlier analysis: capital_gain and capital_loss — box plots + IQR fences
cap_loss_c = get_col('capital_loss')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, label in zip(axes, [cap_gain_c, cap_loss_c], ['Capital Gain', 'Capital Loss']):
    data = df[col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR = Q3 - Q1
    upper_fence = Q3 + 1.5 * IQR
    outliers = (data > upper_fence).sum()
    sns.boxplot(y=data, ax=ax, color='steelblue', flierprops={'alpha': 0.3, 'markersize': 3})
    ax.axhline(upper_fence, color='red', linestyle='--', linewidth=1.5,
               label=f'IQR fence: {upper_fence:.0f}')
    ax.set_title(f'{label}\n{outliers} outliers ({outliers/len(data)*100:.1f}%)',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel(label)
    ax.legend()
plt.suptitle('Outlier Analysis — Capital Gain & Loss', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Decision: Log-transform recommended (log1p) for both capital_gain and capital_loss")
print(f"Zero proportion — capital_gain: {(df[cap_gain_c]==0).mean():.1%} | capital_loss: {(df[cap_loss_c]==0).mean():.1%}")

**Missing Data & Outlier Summary:**
- `occupation` and `workclass` are **co-missing** (correlation ~0.9) — likely the same respondents refused both fields.
- **MNAR test:** Income rate differs meaningfully between missing and non-missing occupation rows — confirming Missing Not At Random. Simple mode imputation may introduce bias; consider a dedicated `Unknown` category.
- `capital_gain` and `capital_loss` are heavily zero-inflated and right-skewed with extreme outliers. **Recommended treatment:** `log1p` transform + consider a binary indicator `has_capital_activity`.

<a id='section-6'></a>
## Section 6 — 15 Annotated Visualizations Gallery

In [ ]:
# Gallery Part 1: Charts 1–6
sex_c = get_col('sex')
marital_c2 = get_col('marital_status')
race_c = get_col('race')
country_c = get_col('native_country')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# 1. Age by income (histogram, hue)
for label, grp in df.groupby(class_c):
    axes[0].hist(grp[age_c].dropna(), bins=30, alpha=0.6, density=True, label=str(label))
axes[0].set_title('1. Age by Income Class', fontweight='bold')
axes[0].set_xlabel('Age'); axes[0].set_ylabel('Density'); axes[0].legend()
axes[0].text(0.98, 0.95, 'Higher income peaks at 40-50', transform=axes[0].transAxes,
             ha='right', va='top', fontsize=8, color='gray')

# 2. Education vs income rate (bar)
edu_rate = df.groupby(education_c)['income_binary'].mean().sort_values()
axes[1].barh(edu_rate.index, edu_rate.values, color=sns.color_palette('husl', len(edu_rate)))
axes[1].set_title('2. Education vs Income Rate', fontweight='bold')
axes[1].set_xlabel('>50K Rate')
axes[1].text(0.98, 0.02, 'Doctorate: ~75% earn >50K', transform=axes[1].transAxes,
             ha='right', fontsize=8, color='gray')

# 3. Occupation vs income rate (sorted)
occ_rate = df.groupby(occupation_c)['income_binary'].mean().sort_values()
axes[2].barh(occ_rate.index, occ_rate.values, color=sns.color_palette('husl', len(occ_rate)))
axes[2].set_title('3. Occupation vs Income Rate (sorted)', fontweight='bold')
axes[2].set_xlabel('>50K Rate')
axes[2].text(0.98, 0.02, 'Exec-managerial tops the list', transform=axes[2].transAxes,
             ha='right', fontsize=8, color='gray')

# 4. Hours/week by income (violin)
sns.violinplot(data=df, x=class_c, y=hours_c, ax=axes[3], palette='husl', inner='box', cut=0)
axes[3].set_title('4. Hours/Week by Income (Violin)', fontweight='bold')
axes[3].set_xlabel('Income Class'); axes[3].set_ylabel('Hours/Week')
axes[3].text(0.98, 0.95, '>50K group works slightly more hours', transform=axes[3].transAxes,
             ha='right', va='top', fontsize=8, color='gray')

# 5. Sex vs income rate (bar)
sex_rate = df.groupby(sex_c)['income_binary'].mean().sort_values()
axes[4].bar(sex_rate.index, sex_rate.values, color=sns.color_palette('husl', 2))
axes[4].set_title('5. Sex vs Income Rate', fontweight='bold')
axes[4].set_ylabel('>50K Rate')
axes[4].text(0.98, 0.95, 'Males ~3x more likely to earn >50K', transform=axes[4].transAxes,
             ha='right', va='top', fontsize=8, color='gray')

# 6. Marital status vs income (stacked bar)
ms_cross = pd.crosstab(df[marital_c2], df[class_c], normalize='index')
ms_cross.plot(kind='barh', stacked=True, ax=axes[5], colormap='RdYlGn')
axes[5].set_title('6. Marital Status vs Income (Stacked)', fontweight='bold')
axes[5].set_xlabel('Proportion'); axes[5].legend(loc='lower right', fontsize=8)
axes[5].text(0.98, 0.02, 'Married-civ-spouse dominates >50K', transform=axes[5].transAxes,
             ha='right', fontsize=8, color='gray')

plt.suptitle('Visualizations Gallery (1–6)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gallery Part 2: Charts 7–12
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# 7. Race vs income rate
race_rate = df.groupby(race_c)['income_binary'].mean().sort_values()
axes[0].barh(race_rate.index, race_rate.values, color=sns.color_palette('husl', len(race_rate)))
axes[0].set_title('7. Race vs Income Rate', fontweight='bold')
axes[0].set_xlabel('>50K Rate')
axes[0].text(0.98, 0.02, 'Significant racial income disparities', transform=axes[0].transAxes,
             ha='right', fontsize=8, color='gray')

# 8. Top-20 countries by count (color = income rate)
top20_countries = df[country_c].value_counts().head(20).index
country_data = df[df[country_c].isin(top20_countries)].groupby(country_c).agg(
    count=(country_c, 'count'), income_rate=('income_binary', 'mean')
).sort_values('count', ascending=True)
colors_c = plt.cm.RdYlGn(country_data['income_rate'])
axes[1].barh(country_data.index, country_data['count'], color=colors_c)
axes[1].set_title('8. Top-20 Countries (color=income rate)', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].text(0.98, 0.02, 'US dominates; high-income from select countries', transform=axes[1].transAxes,
             ha='right', fontsize=8, color='gray')

# 9. Capital gain distribution (log scale)
nonzero_gain = df[df[cap_gain_c] > 0][cap_gain_c]
axes[2].hist(np.log1p(nonzero_gain), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[2].set_title('9. Capital Gain (log1p, non-zero only)', fontweight='bold')
axes[2].set_xlabel('log1p(Capital Gain)'); axes[2].set_ylabel('Count')
axes[2].text(0.98, 0.95, f'Non-zero: {len(nonzero_gain)/len(df)*100:.1f}% of data',
             transform=axes[2].transAxes, ha='right', va='top', fontsize=8, color='gray')

# 10. Age × education heatmap (mean income rate)
df['age_bin'] = pd.cut(df[age_c], bins=[17, 25, 35, 45, 55, 65, 90],
                        labels=['18-25','26-35','36-45','46-55','56-65','66+'])
age_edu_pivot = df.pivot_table(values='income_binary', index='age_bin',
                                columns=edu_num_c, aggfunc='mean')
sns.heatmap(age_edu_pivot, cmap='YlOrRd', ax=axes[3], cbar_kws={'label': 'Income Rate'})
axes[3].set_title('10. Age × Education-Num Income Rate', fontweight='bold')
axes[3].set_xlabel('Education Num'); axes[3].set_ylabel('Age Group')

# 11. Hours worked KDE by income
for label, grp in df.groupby(class_c):
    data = grp[hours_c].dropna()
    kde = stats.gaussian_kde(data)
    x = np.linspace(data.min(), data.max(), 200)
    axes[4].plot(x, kde(x), label=str(label), linewidth=2)
axes[4].set_title('11. Hours Worked — KDE by Income', fontweight='bold')
axes[4].set_xlabel('Hours/Week'); axes[4].set_ylabel('Density'); axes[4].legend()
axes[4].text(0.98, 0.95, '>50K group skewed toward 40-60 hrs', transform=axes[4].transAxes,
             ha='right', va='top', fontsize=8, color='gray')

# 12. Occupation × sex income heatmap (reuse from Section 4)
sns.heatmap(occ_sex_pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[5],
            linewidths=0.3, cbar_kws={'label': '>50K Rate'}, annot_kws={'size': 7})
axes[5].set_title('12. Occupation × Sex Income Rate', fontweight='bold')
axes[5].set_xlabel('Sex'); axes[5].set_ylabel('Occupation')

plt.suptitle('Visualizations Gallery (7–12)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gallery Part 3: Charts 13–15
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 13. Correlation heatmap (compact)
num_cols_small = [age_c, edu_num_c, cap_gain_c, cap_loss_c, hours_c, 'income_binary']
corr_s = df[num_cols_small].corr()
sns.heatmap(corr_s, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('13. Correlation Heatmap', fontweight='bold')
axes[0].text(0.5, -0.15, 'education_num has strongest correlation with income',
             transform=axes[0].transAxes, ha='center', fontsize=8, color='gray')

# 14. Missing values chart
miss_all = df.isnull().sum().sort_values(ascending=True)
miss_all = miss_all[miss_all > 0]
axes[1].barh(miss_all.index, miss_all.values, color=sns.color_palette('husl', len(miss_all)))
axes[1].set_title('14. Missing Values per Column', fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].text(0.98, 0.02, 'Only 3 columns have missing values',
             transform=axes[1].transAxes, ha='right', fontsize=8, color='gray')

# 15. Class imbalance pie
target_c = df[class_c].value_counts()
axes[2].pie(target_c.values, labels=target_c.index, autopct='%1.1f%%',
            colors=sns.color_palette('husl', 2), startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2].set_title('15. Class Imbalance', fontweight='bold')
axes[2].text(0, -1.3, '75:25 imbalance — use stratified splits + F1/AUC metrics',
             ha='center', fontsize=8, color='gray')

plt.suptitle('Visualizations Gallery (13–15)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-7'></a>
## Section 7 — 5 Testable ML Hypotheses

---

### H1: Education is the Strongest Single Predictor of Income

| | |
|---|---|
| **H₀** | Education level is not associated with income class (Cramér's V = 0). |
| **H₁** | Higher education significantly increases the probability of earning >50K. |
| **EDA Evidence** | Cramér's V ≈ 0.34; `education-num` has the highest point-biserial r among numerics; Doctorate holders earn >50K ~75% of the time vs ~16% for HS graduates. |
| **ML Approach** | Logistic Regression with education as sole feature; compare AUC vs null model. Feature importance in Random Forest. |
| **Expected Difficulty** | Low — clear signal, ordinal encoding sufficient. |

---

### H2: Married Individuals Have Significantly Higher Income Rates

| | |
|---|---|
| **H₀** | Marital status is not associated with income (independence). |
| **H₁** | `Married-civ-spouse` individuals are disproportionately likely to earn >50K. |
| **EDA Evidence** | `relationship` and `marital-status` top the Cramér's V ranking. Married-civ-spouse has ~45% income rate vs ~5% for Never-married. Likely a proxy for age + stability. |
| **ML Approach** | Chi-square test for independence; include marital_status in logistic regression and observe coefficient magnitude. |
| **Expected Difficulty** | Low-Medium — strong signal but likely correlated with age and sex (confounding). |

---

### H3: A Binary Capital-Activity Indicator is Highly Predictive

| | |
|---|---|
| **H₀** | Having any capital gain/loss (vs 0) is not predictive of income class. |
| **H₁** | Individuals with non-zero capital transactions earn >50K at much higher rates. |
| **EDA Evidence** | 91% of rows have capital_gain = 0; among those with non-zero gain, >50K rate is dramatically higher. Creating `has_capital = (capital_gain > 0) OR (capital_loss > 0)` is expected to be a strong binary feature. |
| **ML Approach** | Engineer binary feature; compare AUC with vs without it. Logistic regression coefficient test. |
| **Expected Difficulty** | Low — simple feature engineering; very clean signal. |

---

### H4: Long Working Hours (>45 hrs/week) Predict Higher Income

| | |
|---|---|
| **H₀** | Hours worked per week is not associated with income. |
| **H₁** | Individuals working >45 hours/week are more likely to earn >50K. |
| **EDA Evidence** | Violin plots show >50K group has a shifted distribution toward longer hours. KDE by income class shows distinct separation above 45 hrs. Point-biserial r ≈ 0.23. |
| **ML Approach** | Binned hours feature (part-time / standard / overtime); decision tree split analysis. |
| **Expected Difficulty** | Medium — signal exists but is weaker than education; confounded by occupation type. |

---

### H5: The Interaction of Age × Education Outperforms Either Feature Alone

| | |
|---|---|
| **H₀** | Adding an age × education interaction term does not improve model AUC beyond additive effects. |
| **H₁** | Older individuals with high education have a multiplicatively higher income probability than predicted by age or education alone. |
| **EDA Evidence** | Age × education heatmap (Chart 10) shows that high education at age 46-55 has the highest income rate — not captured by either feature independently. |
| **ML Approach** | Logistic Regression: compare AUC of model with age + edu_num vs model with age × edu_num interaction term. Use polynomial features or manual product. |
| **Expected Difficulty** | Medium-High — requires careful feature engineering and cross-validation to avoid overfitting. |

<a id='section-8'></a>
## Section 8 — Professional Report

### Key Findings

- **Class imbalance:** The dataset has a 75:25 imbalance (<=50K vs >50K). A naive classifier that always predicts <=50K achieves 75% accuracy — accuracy alone is a misleading metric; use AUC-ROC or F1.
- **Education is the strongest numeric predictor:** Point-biserial r ≈ 0.33; income rate rises monotonically from 1.8% (Preschool) to ~74% (Doctorate).
- **Marital status / relationship dominate categorically:** Cramér's V > 0.44 for `relationship`; `Married-civ-spouse` has 4× the income rate of `Never-married`.
- **Capital gains are zero-inflated but highly predictive:** 91% of records have `capital_gain = 0`; among non-zero values, >50K rate exceeds 60%. A binary indicator and log1p transform are recommended.
- **Gender income gap is pronounced and consistent:** Males earn >50K at ~3× the rate of females across nearly every occupation. `sex` is statistically predictive but ethically sensitive.
- **~5.7% of occupation and workclass values are missing (MNAR):** Missingness correlates with income class — imputation must be handled carefully to avoid introducing label leakage.
- **Most numeric features are low-correlated (r < 0.3):** Multicollinearity is not a major concern; ensemble methods should perform well without feature selection.

---

### Feature Importance Ranking (EDA Evidence)

| Rank | Feature | Type | Evidence |
|---|---|---|---|
| 1 | relationship / marital-status | Categorical | Cramér's V > 0.44; 4× income rate difference |
| 2 | education / education-num | Both | Cramér's V ≈ 0.34; r ≈ 0.33; monotonic trend |
| 3 | capital-gain (binary/log) | Numeric | Near-perfect separator for non-zero values |
| 4 | occupation | Categorical | Cramér's V ≈ 0.29; Exec-managerial ~50% rate |
| 5 | age | Numeric | r ≈ 0.23; strong violin separation; interaction with education |

---

### Data Limitations

1. **1994 vintage data:** Economic conditions, job categories, and income thresholds have changed dramatically. A $50K threshold in 1994 ≈ ~$100K in 2024 (inflation-adjusted). Models trained on this data should not be deployed in real-world applications without re-labeling.
2. **Sampling bias:** `fnlwgt` is a Census Bureau sampling weight indicating that some demographic groups are underrepresented. Ignoring it may skew model performance for minority subgroups.
3. **Label definition ambiguity:** `>50K` is a broad categorical label — it does not distinguish between $51K and $500K. A regression target or finer-grained buckets would provide more information.
4. **Self-reported categories:** Occupation, workclass, and native-country are self-reported and may be inconsistently labeled. The ~6% missingness in occupation is likely systematic (certain job types are under-represented).

---

### Recommendations for ML Next Steps

1. **Feature engineering pipeline:** Apply `log1p` to `capital_gain` and `capital_loss`, create `has_capital` binary indicator, bin `age` into ordinal groups, and encode `marital_status` as a binary `is_married` flag. Use `ColumnTransformer` + `Pipeline` from sklearn.
2. **Baseline then iterate:** Establish a logistic regression baseline (AUC, F1, precision-recall curve) with stratified 5-fold CV. Then compare against Random Forest and Gradient Boosting (XGBoost/LightGBM). Expect AUC 0.87–0.92 for tree ensembles.
3. **Fairness audit before deployment:** Evaluate model performance separately across `sex` and `race` subgroups. A model with 88% overall AUC may have significantly worse recall for female or minority applicants. Use tools like `fairlearn` to measure and mitigate disparate impact.

<a id='self-check'></a>
## Self-Check Questions

Test your understanding. Try to answer before revealing the solution.

---

**Q1: The dataset is from 1994. What are the key issues if you deployed a model trained on it in 2024?**

<details>
<summary>Show Answer</summary>

**Three main issues:**
1. **Income threshold drift:** $50K/year in 1994 ≈ $100K+ in 2024 (inflation). The label boundary no longer reflects the same socioeconomic reality.
2. **Occupational change:** Entire job categories (tech, gig economy) didn't exist or were negligible in 1994. Categories like "Craft-repair" have shrunk; "Computer/tech" roles were not separately captured.
3. **Demographic and policy shifts:** Labor force participation rates, gender pay gap dynamics, and educational attainment have all changed. A model trained on 1994 patterns may exhibit stale biases. This is called **concept drift** — the relationship between features and the target has shifted over time.
</details>

---

**Q2: `capital_gain` is 0 for 91% of rows. Is this a data quality issue or a real pattern?**

<details>
<summary>Show Answer</summary>

**Real pattern, not a data quality issue.** Most people (especially middle/lower income) do not have significant capital gains — these come from selling stocks, real estate, or business interests. The zero-inflation reflects the actual economic reality: capital gains are concentrated among higher-income individuals.

However, it does create a **modeling challenge** — the distribution is bimodal (zero spike + right-skewed non-zero values). Recommended treatments:
- **Two-part model:** Predict `has_capital` (binary), then predict log-amount for non-zero records.
- **Feature engineering:** `log1p(capital_gain)` + `capital_gain_binary` indicator.
- Do NOT simply remove capital_gain — it's one of the strongest predictors.
</details>

---

**Q3: `sex` is statistically predictive. Should you include it as an ML feature? What are the ethical and legal considerations?**

<details>
<summary>Show Answer</summary>

**This is a core fairness dilemma in ML:**

- **Statistical argument for including it:** Removes it creates a model that performs worse overall, and the residual gender effect may leak through correlated features (occupation, hours-per-week) anyway.
- **Ethical argument against:** If the model is used for lending, hiring, or benefits decisions, using sex as a feature directly encodes historical discrimination — the wage gap in 1994 reflects systemic inequality, not an innate productivity difference.
- **Legal context:** In many jurisdictions, using protected attributes (sex, race, religion) in automated decision systems is prohibited under anti-discrimination law (e.g., ECOA in the US, GDPR in the EU).

**Best practice:** Remove `sex` and `race` as direct features, but audit the model for disparate impact using these as protected attributes. Tools: `fairlearn`, `aequitas`, `AI Fairness 360`.
</details>

---

**Q4: The dataset has 75% class imbalance. How does this affect a naive classifier and what metrics should you use?**

<details>
<summary>Show Answer</summary>

**The naive classifier problem:**
A classifier that always predicts `<=50K` (the majority class) achieves **75% accuracy** with zero predictive value. This is why accuracy is a misleading metric for imbalanced datasets.

**Preferred metrics:**
- **AUC-ROC:** Measures ranking ability across all thresholds; unaffected by class distribution.
- **F1-score (minority class):** Harmonic mean of precision and recall for the `>50K` class.
- **Precision-Recall AUC:** More informative than ROC when the positive class is rare.
- **Matthews Correlation Coefficient (MCC):** Balanced metric even for imbalanced classes.

**Mitigation strategies:** Stratified train/test splits, class_weight='balanced' in sklearn, SMOTE for oversampling, or cost-sensitive learning.
</details>

---

**Q5: `fnlwgt` is a Census sample weight. Should you use it as an ML feature?**

<details>
<summary>Show Answer</summary>

**Generally: No, you should not use `fnlwgt` as a direct ML feature.**

`fnlwgt` (final weight) represents how many people in the US population that individual's record is meant to represent — it is a **survey design artifact**, not a characteristic of the individual. Including it as a feature would:
1. Introduce noise with no causal connection to income.
2. Cause model behavior that's hard to interpret or explain.
3. Potentially leak information about sampling strata in ways that don't generalize.

**When it IS useful:** As a **sample weight** in model training (the `sample_weight` parameter in sklearn), so that predictions are representative of the full US population rather than the (biased) sample distribution. This is particularly important for fairness and demographic parity evaluations.
</details>